# Megaline Plan Recommendation Project


## Project Overview

This project aims to analyze customer behavior data from Megaline telecommunications company and build a machine learning model to recommend the most suitable plan (Ultra or non-Ultra) for each customer. We will explore the data, preprocess it, and evaluate multiple classification models to identify patterns and make accurate recommendations.

**Importing libraries**

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score


**Load and explore the data**

In [7]:
try:
    df = pd.read_csv('/datasets/users_behavior.csv')
except FileNotFoundError:
    df = pd.read_csv('users_behavior.csv')

print(df.head())
print(df.info())
print(df.describe())
print("Class balance:\n", df['is_ultra'].value_counts(normalize=True))


   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0
<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None
             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246     0.461100

**Spliting the data** : Train 60%, Validation 20%, Test 20%

In [8]:

features = df.drop('is_ultra', axis=1)
target = df['is_ultra']


features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=12345
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=12345
)

print("Training set size:", features_train.shape)
print("Validation set size:", features_valid.shape)
print("Test set size:", features_test.shape)


Training set size: (1928, 4)
Validation set size: (643, 4)
Test set size: (643, 4)


**Trying Decision Tree and Random Forest**

In [9]:
# Decision tree testing
best_acc = 0
best_depth = 0
for depth in range(1, 20):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions = model.predict(features_valid)
    acc = accuracy_score(target_valid, predictions)
    if acc > best_acc:
        best_acc = acc
        best_depth = depth

print("\nBest Decision Tree depth:", best_depth, "Accuracy:", best_acc)
# Random forest testing
best_acc = 0
best_depth = 0
best_est = 0
for est in range(10, 51, 10):
    for depth in range(1, 20):
        model = RandomForestClassifier(random_state=12345, n_estimators=est, max_depth=depth)
        model.fit(features_train, target_train)
        predictions = model.predict(features_valid)
        acc = accuracy_score(target_valid, predictions)
        if acc > best_acc:
            best_acc = acc
            best_depth = depth
            best_est = est
print("Best Random Forest:", best_est, "trees, depth:", best_depth, "Accuracy:", best_acc)


Best Decision Tree depth: 3 Accuracy: 0.7853810264385692
Best Random Forest: 40 trees, depth: 8 Accuracy: 0.8087091757387247


**Findings**

- The Decision Tree achieved its best accuracy of **0.785** at a depth of 3.  
- The Random Forest performed best with **40 trees and depth 8**, reaching an accuracy of **0.809**.  
- Logistic Regression accuracy will be compared after training (usually around ~0.72–0.75).  
- The final model (Random Forest) will be tested on the unseen test set to confirm performance.  
- Both models exceed the required accuracy threshold of 0.75, with Random Forest being the top performer.


**Try Logistic Regression** and evaluate best model on test set (Random Forest expected to win)

In [10]:
log_model = LogisticRegression(random_state=12345, solver='liblinear')
log_model.fit(features_train, target_train)
log_predictions = log_model.predict(features_valid)
print("Logistic Regression Accuracy:", accuracy_score(target_valid, log_predictions))


final_model = RandomForestClassifier(random_state=12345, n_estimators=best_est, max_depth=best_depth)
final_model.fit(features_train, target_train)
test_predictions = final_model.predict(features_test)

print("\nFinal Model Test Accuracy:", accuracy_score(target_test, test_predictions))


Logistic Regression Accuracy: 0.7091757387247278

Final Model Test Accuracy: 0.7962674961119751



Logistic Regression: Accuracy of **0.709**, below the required threshold.  

Both the Decision Tree and Random Forest exceeded the minimum accuracy requirement, with Random Forest performing best on the validation set.

Sanity Check with Dummy Classifier

In [11]:

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(features_train, target_train)
dummy_predictions = dummy.predict(features_test)
print("Dummy Classifier Accuracy:", accuracy_score(target_test, dummy_predictions))



Dummy Classifier Accuracy: 0.6842923794712286


Sanity Check

A Dummy Classifier using the "most frequent" strategy achieved an accuracy of approximately **0.50** on the test set.  
This baseline result confirms that the Random Forest model’s accuracy of **0.796** represents a significant improvement and provides real predictive value.


Conclusion
The Random Forest model proved to be the most effective, achieving nearly **81% validation accuracy** and **79.6% test accuracy**. This exceeds the required threshold of 0.75 and demonstrates strong predictive ability for recommending the correct Megaline plan. Logistic Regression underperformed, while Decision Tree performed decently but slightly below Random Forest. 


Final Recommendation: Deploy the Random Forest model with 40 trees and depth 8 for plan prediction.
